# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. `1 1.3333333333333333`: пропуск один, `fillna(0)` даёт [1, 0, 3], среднее 4 / 3.
2. `2`: после `strip` и `lower` получается «a», «a», «b», то есть два разных значения.
3. `3`: `drop_duplicates()` возвращает новую таблицу, а результат не сохранён, поэтому `df` не изменился.
4. `1005`: пробел убран, «1000» + «5» как числа дают 1005.
5. `[9, 10]`: 30 сентября и 1 октября, `.dt.month` берёт номер месяца.


In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [2]:
# 1
import pandas as pd

# Все написания городов → одно правильное название
CITY_MAP = {
    "алматы": "Алматы", "АЛМАТЫ": "Алматы", "Almaty": "Алматы",
    "астана": "Астана", "Astana": "Астана",
    "шымкент": "Шымкент", "Shymkent": "Шымкент",
}

def clean(path):
    df = pd.read_csv(path)
    print("Прочитали:", len(df))

    # 1. Суммы "15 990" → числа
    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print("Суммы в числа:", len(df))

    # 2. Города: убираем пробелы по краям и переводим через словарь
    df["city"] = df["city"].str.strip().replace(CITY_MAP)
    print("Города через словарь:", len(df), "| городов:", df["city"].nunique())

    # 3. Дубли
    df = df.drop_duplicates()
    print("Без дублей:", len(df))

    # 4. Даты: текст → datetime, отдельно месяц
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["month"] = df["order_date"].dt.month
    print("Даты:", len(df))

    # 5. Без тестовых заказов (отменённые оставляем)
    df = df[df["customer_id"] != "TEST"]
    print("Без TEST:", len(df))

    # Проверки
    assert df["amount"].isna().sum() == 0, "в amount есть NaN"
    assert df["city"].isna().sum() == 0, "в city есть NaN"
    assert df["order_id"].is_unique, "order_id не уникален"
    return df

df = clean("orders.csv")
print()
print("Итого строк:", len(df))
print(df["city"].value_counts())


Прочитали: 5463
Суммы в числа: 5463
Города через словарь: 5463 | городов: 6
Без дублей: 5283
Даты: 5283
Без TEST: 5280

Итого строк: 5280
city
Алматы       2116
Астана       1339
Шымкент       776
Караганда     519
Актобе        387
Атырау        143
Name: count, dtype: int64


**Вывод для менеджера:** после очистки осталось **5 280 заказов** из 5 463. Убраны 180 дублей и 3 тестовых заказа, 14 написаний городов сведены к 6. Отменённые заказы оставлены, они нужны для доли отмен.


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [3]:
# 2
paid = df[df["status"] == "оплачен"]

def change(data):
    """Выручка за сентябрь и октябрь и изменение в %."""
    sep = data.loc[data["month"] == 9, "amount"].sum()
    octo = data.loc[data["month"] == 10, "amount"].sum()
    return sep, octo, (octo - sep) / sep * 100

sep, octo, pct = change(paid)
print(f"Все города: сентябрь {sep:,.0f} ₸ → октябрь {octo:,.0f} ₸, изменение {pct:.1f} %")
print()
for city in paid["city"].unique():
    sep, octo, pct = change(paid[paid["city"] == city])
    print(f"{city:10} сентябрь {sep:>12,.0f} → октябрь {octo:>12,.0f}   {pct:6.1f} %")


Все города: сентябрь 35,243,410 ₸ → октябрь 34,419,520 ₸, изменение -2.3 %

Алматы     сентябрь   14,501,070 → октябрь   14,643,020      1.0 %
Астана     сентябрь    9,034,220 → октябрь    8,850,530     -2.0 %
Караганда  сентябрь    3,648,520 → октябрь    3,371,680     -7.6 %
Шымкент    сентябрь    4,558,650 → октябрь    4,044,010    -11.3 %
Атырау     сентябрь      923,990 → октябрь      969,860      5.0 %
Актобе     сентябрь    2,576,960 → октябрь    2,540,420     -1.4 %


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [ ]:
# 3
# Повторяем шаги очистки по одному и после каждого считаем «падение выручки» (change из задания 2)
step = pd.read_csv("orders.csv")
step["amount"] = pd.to_numeric(step["amount"].str.replace(" ", ""))
step["month"] = pd.to_datetime(step["order_date"]).dt.month

def drop_pct(data):
    return round(change(data[data["status"] == "оплачен"])[2], 1)

n0 = len(step)
print("Сырые данные (суммы в числа):", n0, "строк, изменение", drop_pct(step), "%")

step["city"] = step["city"].str.strip().replace(CITY_MAP)

print("Города через словарь: убрано", n0 - len(step), "| изменение", drop_pct(step), "%")

n1 = len(step)
step = step.drop_duplicates()
print("Без дублей: убрано", n1 - len(step), "| изменение", drop_pct(step), "%")

n2 = len(step)
step = step[step["customer_id"] != "TEST"]
print("Без TEST: убрано", n2 - len(step), "| изменение", drop_pct(step), "%")


Сырые данные (суммы в числа): 5463 строк, изменение -22.7 %
Города через словарь: убрано 0 | изменение -22.7 %
Без дублей: убрано 180 | изменение -19.6 %
Без TEST: убрано 3 | изменение -2.3 %


| Шаг очистки | Убрано строк | Падение выручки |
|---|---|---|
| Сырые данные (суммы в числа) | 0 | −22,7% |
| Города через словарь | 0 | −22,7% |
| Удаление дублей | 180 | −19,6% |
| Даты в datetime | 0 | −19,6% |
| Удаление TEST | 3 | **−2,3%** |

**Вывод для менеджера:** сильнее всего цифру исказили **3 тестовых заказа**. Всего 3 строки, но из-за них «падение» было на 17 п.п. страшнее, чем на самом деле.


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [ ]:
# 4
na = df["delivery_days"].isna()
print("Пропусков в delivery_days:", na.sum(), "из", len(df))
print()
print("По городам:")
print(df.loc[na, "city"].value_counts())
print()
print("По статусам:")

print(df.loc[na, "status"].value_counts())
print()

sh_days = df.loc[df["city"] == "Шымкент", "delivery_days"]
print("Шымкент: заказов", len(sh_days), "| пропусков", sh_days.isna().sum(), "| медиана", sh_days.median())
a = (sh_days.dropna() == 2).mean()

b = (sh_days.fillna(sh_days.median()) == 2).mean()
c = (sh_days.fillna(0) == 2).mean()
print(f"Доля доставки за 2 дня: (а) dropna {a:.1%} | (б) fillna(медиана) {b:.1%} | (в) fillna(0) {c:.1%}")


Пропусков в delivery_days: 158 из 5280

По городам:
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64

По статусам:
status
оплачен    144
отменён     14
Name: count, dtype: int64

Шымкент: заказов 776 | пропусков 22 | медиана 5.0
Доля доставки за 2 дня: (а) dropna 14.9% | (б) fillna(медиана) 14.4% | (в) fillna(0) 14.4%


**Вывод для менеджера:** срок доставки не указан у **158 заказов (3%)**. Пропуски распределены по городам примерно пропорционально числу заказов, 144 из них у оплаченных заказов. В Шымкенте доля доставки за 2 дня: **14,9%** при dropna, 14,4% при заполнении медианой и 14,4% при заполнении нулём. Честный вариант **dropna**: считаем только заказы, срок которых известен. Заполнение медианой (5 дней) приписывает заказам выдуманный срок, а 0 дней вообще невозможная доставка. Оба варианта искажают распределение сроков.


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [ ]:
# 5
fixed = pd.read_csv("orders.csv")

fixed = fixed.drop_duplicates()

fixed["city"] = fixed["city"].str.strip().replace(CITY_MAP)
print(fixed["city"].nunique(), len(fixed))


6 5283


**Вывод для менеджера:** исправленная очистка даёт **6 городов и 5 283 строки** (до удаления TEST). AI ошибся дважды: не сохранил результат удаления дублей и думал, что маленькие буквы объединят «Almaty» и «Алматы».


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [7]:
# 6
days = df["order_date"].dt.day_name().value_counts()
print(days)
print()
busiest, quietest = days.index[0], days.index[-1]
less = (days.iloc[0] - days.iloc[-1]) / days.iloc[0] * 100
print(f"Больше всего заказов: {busiest} ({days.iloc[0]}), меньше всего: {quietest} ({days.iloc[-1]})")
print(f"В {quietest} заказов меньше на {less:.1f} %")


order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64

Больше всего заказов: Thursday (786), меньше всего: Monday (684)
В Monday заказов меньше на 13.0 %


**Вывод для менеджера:** заказы распределены по будням ровно (774–786 в день). Меньше всего заказов в **понедельник (684), на 13% меньше**, чем в самый загруженный четверг. В воскресенье тоже мало (697). В понедельник и воскресенье можно ставить меньше курьеров, а со вторника по субботу держать полный состав.


## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [8]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [9]:
# 8